In [3]:
import pandas as pd
import spacy
import re

nlp = spacy.load("en_core_web_sm")

df = pd.read_csv("../../dataset.csv")
BAD_ORGS = {
    "Reuters",
    "REUTERS",
    "Reuters Staff",
    "CNBC",
    "BBC",
    "Fortune",
    "PRNewswire",
    "Globe Newswire",
    "Eikon",
    "SEDAR",
}

def clean_company_name(name):
    name = str(name).strip()
    name = re.sub(r"\s+", " ", name)
    name = name.strip(".,;:()[]{}\"'")
    return name

def is_probably_company(name):
    if not name:
        return False

    if name in BAD_ORGS:
        return False

    if len(name) < 2:
        return False

    generic_terms = {
        "Reporting",
        "Editing",
        "Source",
        "Conference Call",
        "Forward-Looking Statements",
        "Non-GAAP Financial Measures",
    }

    if name in generic_terms:
        return False

    return True


def extract_companies(text):
    if pd.isna(text):
        return []

    doc = nlp(str(text))

    companies = []

    for ent in doc.ents:
        if ent.label_ == "ORG":
            company = clean_company_name(ent.text)

            if is_probably_company(company):
                companies.append(company)

    seen = set()
    unique_companies = []

    for company in companies:
        key = company.lower()

        if key not in seen:
            seen.add(key)
            unique_companies.append(company)

    return unique_companies


rows = []

for article_id, row in df.iterrows():
    companies = extract_companies(row["text"])
    if article_id % 10000 == 0:
        print(f"Processing article {article_id} / {len(df)}")
    for company in companies:
        rows.append({
            "article_id": article_id,
            "date": row["date"],
            "title": row["title"],
            "company": company,
            "url": row["url"]
        })

company_df = pd.DataFrame(rows)
article_company_lists = (company_df.groupby(["article_id", "date", "title", "url"], as_index=False).agg({"company": list}))
article_company_lists.to_csv("companies_in_articles.csv", index=False)

print(article_company_lists.head(20))
print(f"Saved {len(article_company_lists)} article-company rows.")

Processing article 0 / 50000
Processing article 10000 / 50000
Processing article 20000 / 50000
Processing article 30000 / 50000
Processing article 40000 / 50000
    article_id        date                                              title  \
0            0  2018-04-26  McLaren review F1 technical operations, Goss m...   
1            1  2018-05-15  ADDvantage Technologies Announces Financial Re...   
2            2  2018-04-10  UPDATE 2-Vitol's African venture Vivo to float...   
3            3  2018-03-19  BRIEF-Pareteum Awarded $2.4 Mln Contract From ...   
4            4  2018-03-29  Jaguar Mining Reports 2017 Fourth Quarter and ...   
5            5  2018-02-23  Puerto Rico governor announces independent pro...   
6            6  2018-05-04       Alaska Air Group Declares Quarterly Dividend   
7            7  2018-05-08                         BioCryst Reports Financial   
8            8  2018-05-31  Higher Ed Partners, UK Appoints New Board Members   
9            9  2018-02-05  N